In [ ]:
from IPython.core.magic import register_cell_magic
import time

@register_cell_magic
def time_all(line, cell):
    start = time.time()
    shell = get_ipython()
    shell.run_cell(cell)
    end = time.time()
    print(f"[Cell Execution Time: {end - start:.4f} seconds]")

# To use it on a cell, just put %%time_all at the top


Cell 1: Download Historical Prices & Save to Parquet

In [1]:
import pandas as pd
import yfinance as yf
import os
import warnings

# Suppress yfinance warnings for cleaner output
warnings.filterwarnings("ignore")

# --- FILE PATHS ---
work_dir = r"C:\Users\sudha\Downloads\Wilshire 5000 Index Fund"
ref_file = os.path.join(work_dir, "company_ticker_reference_updated.parquet")
prices_file = os.path.join(work_dir, "historical_prices_2015_2026.parquet")

# 1. Load your master reference file
ref_df = pd.read_parquet(ref_file)

# 2. Extract valid tickers (filter out 'Not Found', 'N/A', etc.)
invalid_flags = ['Not Found', 'N/A', 'Failed']
valid_tickers_series = ref_df[~ref_df['yfinance identifier'].str.contains('|'.join(invalid_flags), na=False)]['yfinance identifier']
tickers_list = [str(t).strip() for t in valid_tickers_series.unique() if pd.notna(t)]

print(f"Starting download for {len(tickers_list)} valid tickers...")

# 3. Bulk Download from yfinance (this respects basic rate limits via threads)
# We use auto_adjust=False so we keep both Close and Adj Close
raw_data = yf.download(
    tickers_list, 
    start="2015-01-01", 
    end="2026-09-30", 
    group_by='ticker', 
    threads=True,
    auto_adjust=False
)

# 4. Restructure the MultiIndex DataFrame into a flat table
print("\nRestructuring data for Parquet format...")
# Stack shifts the tickers from columns to rows
stacked_data = raw_data.stack(level=0).rename_axis(['Date', 'Ticker']).reset_index()

# Drop rows where all price data is NaN (e.g., weekends, holidays, or before IPO)
clean_data = stacked_data.dropna(subset=['Close', 'Adj Close'], how='all')

# 5. Save to Parquet
clean_data.to_parquet(prices_file, index=False)
print(f"✅ Successfully saved historical data to: {prices_file}")

Starting download for 1921 valid tickers...


[*                      3%                       ]  60 of 1921 completed$1PGM.XD: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-09-30)
[*****                 10%                       ]  183 of 1921 completed$HDID.XD: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-09-30)
[*****                 10%                       ]  186 of 1921 completed$SCOP0=F: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-09-30)
[*******               15%                       ]  288 of 1921 completed$XY6D.XD: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-09-30)
[*********             19%                       ]  371 of 1921 completed$EFR.NE: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-09-30)
[**********            20%                       ]  383 of 1921 completed$KHNZD.XD: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-09-30)
[***********           22%                       ]  416 of 1921 completed$GPS: poss


Restructuring data for Parquet format...
✅ Successfully saved historical data to: C:\Users\sudha\Downloads\Wilshire 5000 Index Fund\historical_prices_2015_2026.parquet


Cell 2: Read Parquet & Generate Summary

In [2]:
import pandas as pd
import os

prices_file = r"C:\Users\sudha\Downloads\Wilshire 5000 Index Fund\historical_prices_2015_2026.parquet"

# 1. Read from the saved Parquet file
print("Loading data from Parquet...")
df_prices = pd.read_parquet(prices_file)

# 2. Summarize days of data per ticker
summary = df_prices.groupby('Ticker').size().reset_index(name='Days of Data')
summary = summary.sort_values(by='Days of Data', ascending=False).reset_index(drop=True)

# Calculate some overall stats
max_days = summary['Days of Data'].max()
min_days = summary['Days of Data'].min()
total_tickers = len(summary)
full_history_tickers = len(summary[summary['Days of Data'] == max_days])

print(f"\n--- DATA SUMMARY ---")
print(f"Total Unique Tickers in File: {total_tickers}")
print(f"Maximum Trading Days (Full History): {max_days}")
print(f"Tickers with Full History (since Jan 2015): {full_history_tickers}")
print(f"Minimum Trading Days (Recent IPOs): {min_days}\n")

print("Top 5 Tickers by Days of Data:")
print(summary.head())
print("\nBottom 5 Tickers by Days of Data (Likely recent IPOs):")
print(summary.tail())

Loading data from Parquet...

--- DATA SUMMARY ---
Total Unique Tickers in File: 1507
Maximum Trading Days (Full History): 2985
Tickers with Full History (since Jan 2015): 6
Minimum Trading Days (Recent IPOs): 59

Top 5 Tickers by Days of Data:
   Ticker  Days of Data
0  62C.SG          2985
1   BOX.F          2985
2   AXV.F          2985
3  SCL.DU          2985
4  JB1.MU          2985

Bottom 5 Tickers by Days of Data (Likely recent IPOs):
      Ticker  Days of Data
1502     FPS           163
1503    SUNB           147
1504    TRAX           123
1505    FDXF            87
1506  801.DE            59


Cell 3: Fetch Sector Categorization (Rate-Limit Safe)

In [1]:
import pandas as pd
import yfinance as yf
import os
import time
from tqdm.auto import tqdm # Progress bar

work_dir = r"C:\Users\sudha\Downloads\Wilshire 5000 Index Fund"
prices_file = os.path.join(work_dir, "historical_prices_2015_2026.parquet")
sectors_file = os.path.join(work_dir, "ticker_sectors.csv")

# Get unique tickers from the saved prices file
df_prices = pd.read_parquet(prices_file, columns=['Ticker'])
unique_tickers = df_prices['Ticker'].unique().tolist()

# Load existing sector data if we already started this process
if os.path.exists(sectors_file):
    sector_df = pd.read_csv(sectors_file)
    existing_tickers = sector_df['Ticker'].tolist()
    sector_dict = dict(zip(sector_df['Ticker'], sector_df['Sector']))
else:
    sector_dict = {}
    existing_tickers = []

# Find which tickers we still need to fetch
tickers_to_fetch = [t for t in unique_tickers if t not in existing_tickers]

print(f"Total tickers: {len(unique_tickers)}")
print(f"Already fetched: {len(existing_tickers)}")
print(f"Remaining to fetch: {len(tickers_to_fetch)}")

if len(tickers_to_fetch) > 0:
    print("\nFetching sector data (this will take a while to respect API limits)...")
    
    # Batch save every 20 tickers so we don't lose data if it crashes
    save_counter = 0 
    
    for ticker in tqdm(tickers_to_fetch, desc="Fetching Sectors"):
        try:
            # Fetch info
            info = yf.Ticker(ticker).info
            sector = info.get('sector', 'Unknown')
            sector_dict[ticker] = sector
            
        except Exception as e:
            sector_dict[ticker] = 'Failed'
            
        # Respect API rate limits (Wait 0.5 seconds between calls)
        time.sleep(0.5)
        
        # Save incrementally
        save_counter += 1
        if save_counter % 20 == 0:
            pd.DataFrame(list(sector_dict.items()), columns=['Ticker', 'Sector']).to_csv(sectors_file, index=False)
            
    # Final save
    pd.DataFrame(list(sector_dict.items()), columns=['Ticker', 'Sector']).to_csv(sectors_file, index=False)
    print(f"\n✅ Finished fetching sectors. Saved to: {sectors_file}")
else:
    print("✅ All sectors already fetched!")

# Optional: Print a quick sector breakdown
final_sector_df = pd.read_csv(sectors_file)
print("\n--- SECTOR BREAKDOWN ---")
print(final_sector_df['Sector'].value_counts())

Total tickers: 1507
Already fetched: 0
Remaining to fetch: 1507

Fetching sector data (this will take a while to respect API limits)...


Fetching Sectors:   0%|          | 0/1507 [00:00<?, ?it/s]


✅ Finished fetching sectors. Saved to: C:\Users\sudha\Downloads\Wilshire 5000 Index Fund\ticker_sectors.csv

--- SECTOR BREAKDOWN ---
Sector
Technology                250
Financial Services        243
Industrials               226
Healthcare                222
Consumer Cyclical         182
Real Estate                99
Basic Materials            62
Consumer Defensive         61
Energy                     59
Communication Services     48
Utilities                  44
Unknown                     9
Name: count, dtype: int64


Cell 4: Fetching sharesOutstanding and floatShares

In [1]:
import pandas as pd
import yfinance as yf
import os
import time
from tqdm.auto import tqdm

# --- FILE PATHS ---
work_dir = r"C:\Users\sudha\Downloads\Wilshire 5000 Index Fund"
ref_file = os.path.join(work_dir, "company_ticker_reference_updated.parquet")
shares_csv = os.path.join(work_dir, "ticker_shares.csv")
shares_parquet = os.path.join(work_dir, "ticker_shares.parquet")

# 1. Get valid tickers from your master reference file
ref_df = pd.read_parquet(ref_file)
invalid_flags = ['Not Found', 'N/A', 'Failed', 'Manual Override']
valid_tickers = ref_df[~ref_df['yfinance identifier'].str.contains('|'.join(invalid_flags), na=False)]['yfinance identifier'].unique().tolist()

# 2. Check if we already started downloading to resume progress
if os.path.exists(shares_csv):
    existing_df = pd.read_csv(shares_csv)
    existing_tickers = existing_df['Ticker'].tolist()
    shares_data = existing_df.to_dict('records')
else:
    existing_tickers = []
    shares_data = []

tickers_to_fetch = [t for t in valid_tickers if t not in existing_tickers]

print(f"Total tickers: {len(valid_tickers)}")
print(f"Already fetched: {len(existing_tickers)}")
print(f"Remaining to fetch: {len(tickers_to_fetch)}\n")

# 3. Fetch Data
if len(tickers_to_fetch) > 0:
    save_counter = 0
    
    for ticker in tqdm(tickers_to_fetch, desc="Fetching Share Data"):
        try:
            info = yf.Ticker(ticker).info
            shares_out = info.get('sharesOutstanding', None)
            float_shares = info.get('floatShares', None)
            
            shares_data.append({
                'Ticker': ticker,
                'Shares Outstanding': shares_out,
                'Float Shares': float_shares
            })
        except Exception as e:
            shares_data.append({
                'Ticker': ticker,
                'Shares Outstanding': None,
                'Float Shares': None
            })
            
        time.sleep(0.3) # Polite delay to avoid rate limits
        
        # Save every 20 tickers
        save_counter += 1
        if save_counter % 20 == 0:
            pd.DataFrame(shares_data).to_csv(shares_csv, index=False)
            
    # Final Save to CSV and Parquet
    final_df = pd.DataFrame(shares_data)
    final_df.to_csv(shares_csv, index=False)
    final_df.to_parquet(shares_parquet, index=False)
    print(f"✅ Finished! Saved to {shares_parquet}")
else:
    print("✅ All share data already fetched!")

Total tickers: 1921
Already fetched: 0
Remaining to fetch: 1921



Fetching Share Data:   0%|          | 0/1921 [00:00<?, ?it/s]

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BK"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CPRX"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CCRN"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: LANC"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NSA"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: SEM"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: SILA"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbo

✅ Finished! Saved to C:\Users\sudha\Downloads\Wilshire 5000 Index Fund\ticker_shares.parquet


Cell 5: Calculating Market Cap & Classify (Mega to Nano)

In [2]:
import pandas as pd
import os
import numpy as np

# --- FILE PATHS ---
work_dir = r"C:\Users\sudha\Downloads\Wilshire 5000 Index Fund"
prices_file = os.path.join(work_dir, "historical_prices_2015_2026.parquet")
shares_file = os.path.join(work_dir, "ticker_shares.parquet")
output_file = os.path.join(work_dir, "market_cap_classifications.parquet")

# 1. Load Data
print("Loading data...")
df_prices = pd.read_parquet(prices_file)
df_shares = pd.read_parquet(shares_file)

# 2. Get the *latest* price for each ticker to calculate current Market Cap
# Sort by date and take the last row for each ticker
latest_prices = df_prices.sort_values('Date').groupby('Ticker').last().reset_index()
latest_prices = latest_prices[['Ticker', 'Close', 'Date']].rename(columns={'Close': 'Latest Price', 'Date': 'Price Date'})

# 3. Merge Prices and Shares
df_mcap = pd.merge(latest_prices, df_shares, on='Ticker', how='inner')

# 4. Calculate Market Cap
df_mcap['Market Cap (USD)'] = df_mcap['Latest Price'] * df_mcap['Shares Outstanding']
df_mcap['Free Float Market Cap (USD)'] = df_mcap['Latest Price'] * df_mcap['Float Shares']

# 5. Define Market Cap Classification Logic
def categorize_market_cap(mcap):
    if pd.isna(mcap):
        return 'Unknown'
    elif mcap > 200_000_000_000:       # > $200 Billion
        return 'Mega-Cap'
    elif mcap >= 10_000_000_000:       # $10B - $200B
        return 'Large-Cap'
    elif mcap >= 2_000_000_000:        # $2B - $10B
        return 'Mid-Cap'
    elif mcap >= 250_000_000:          # $250M - $2B
        return 'Small-Cap'
    elif mcap >= 50_000_000:           # $50M - $250M
        return 'Micro-Cap'
    else:                              # < $50 Million
        return 'Nano-Cap'

print("Applying classifications...")
df_mcap['Market Cap Category'] = df_mcap['Market Cap (USD)'].apply(categorize_market_cap)

# 6. Save and Summarize
df_mcap.to_parquet(output_file, index=False)
df_mcap.to_csv(output_file.replace('.parquet', '.csv'), index=False) # Saving a CSV for human viewing too

print(f"✅ Market Cap categorizations calculated and saved to {output_file}\n")

# Display a summary table
print("--- SUMMARY OF MARKET CAP CATEGORIES ---")
summary_counts = df_mcap['Market Cap Category'].value_counts()
print(summary_counts)

# Let's peek at the Mega-Caps
mega_caps = df_mcap[df_mcap['Market Cap Category'] == 'Mega-Cap']
print(f"\nFound {len(mega_caps)} Mega-Cap stocks. Here are a few:")
print(mega_caps[['Ticker', 'Market Cap Category', 'Market Cap (USD)']].head())

Loading data...
Applying classifications...
✅ Market Cap categorizations calculated and saved to C:\Users\sudha\Downloads\Wilshire 5000 Index Fund\market_cap_classifications.parquet

--- SUMMARY OF MARKET CAP CATEGORIES ---
Market Cap Category
Mid-Cap      605
Large-Cap    477
Small-Cap    356
Mega-Cap      42
Micro-Cap     15
Unknown       10
Nano-Cap       2
Name: count, dtype: int64

Found 42 Mega-Cap stocks. Here are a few:
    Ticker Market Cap Category  Market Cap (USD)
15    ABBV            Mega-Cap      4.652643e+11
79    AMAT            Mega-Cap      4.063298e+11
82     AMD            Mega-Cap      9.918429e+11
97    AMZN            Mega-Cap      2.660660e+12
101   ANET            Mega-Cap      2.558520e+11
